In [ ]:
import torch
import torchvision.models
from torchvision.datasets import ImageNet
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from pytorch_grad_cam import HiResCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image
from torchvision.transforms.functional import to_pil_image
from torchvision import transforms as transforms
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

device = torch.device("cuda")

class LinearClassifier(nn.Module):
    """Linear layer to train on top of frozen features"""

    def __init__(self, dim, num_labels=1000):
        super(LinearClassifier, self).__init__()
        self.num_labels = num_labels
        self.linear = nn.Linear(dim, num_labels)
        self.linear.weight.data.normal_(mean=0.0, std=0.01)
        self.linear.bias.data.zero_()

    def forward(self, x):
        # flatten
        x = x.view(x.size(0), -1)

        # linear layer
        return self.linear(x)

In [ ]:
def load_ddp_checkpoint(model, checkpoint_path):
    state_dict = torch.load(checkpoint_path)
    new_state_dict = {k.replace('module.', ''): v for k, v in state_dict.items()}
    model.load_state_dict(new_state_dict)
    return model

In [ ]:
steps = ["5_3400", "10_6800", "20_13400", "30_20200", "99_66799"]
models_steps = []
linear_steps = []
for step in steps:
    resnet = torchvision.models.resnet50(weights=None)
    embed_dim = resnet.fc.weight.shape[1]
    resnet.fc = nn.Identity()
    resnet.load_state_dict(torch.load(f"/models/jhkim/bt-imagenet/{step}.pth"))
    models_steps.append(resnet)
    linear = LinearClassifier(embed_dim)
    load_ddp_checkpoint(linear, f"/models/jhkim/bt-imagenet/{step}/final_checkpoint.pth")
    linear_steps.append(linear)

In [ ]:
import xml.etree.ElementTree as ET
from PIL import Image, ImageDraw
import os

def parse_bbox_annotation(xml_path):
    if not os.path.exists(xml_path):
        return []

    tree = ET.parse(xml_path)
    root = tree.getroot()

    bboxes = []
    for obj in root.findall('object'):
        bbox = obj.find('bndbox')
        xmin = int(bbox.find('xmin').text)
        ymin = int(bbox.find('ymin').text)
        xmax = int(bbox.find('xmax').text)
        ymax = int(bbox.find('ymax').text)

        bboxes.append([xmin, ymin, xmax, ymax])

    return bboxes

def mask_object_regions(image_path, xml_path):
    img = Image.open(image_path).convert('RGB')

    bboxes = parse_bbox_annotation(xml_path)

    if bboxes:
        draw = ImageDraw.Draw(img)
        for bbox in bboxes:
            xmin, ymin, xmax, ymax = bbox
            draw.rectangle([xmin, ymin, xmax, ymax], fill='black')

    return img

val_transform = transforms.Compose([
    transforms.Resize(256, interpolation=3),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),
])

def denormalize(tensor):
    mean = torch.tensor((0.485, 0.456, 0.406)).view(3, 1, 1).to(tensor.device)
    std = torch.tensor((0.229, 0.224, 0.225)).view(3, 1, 1).to(tensor.device)
    denorm = tensor * std + mean
    return torch.clamp(denorm, 0, 1)

dataset_test = ImageNet("/home/dataset/imagenet", split="val", transform=val_transform)
bbox_dir = "/home/dataset/imagenet/ILSVRC/Annotations/CLS-LOC/val"

In [ ]:
classes = {0: "tench", 678: "neck brace"}
samples_per_class = 10

class_indices = {cls: [] for cls in classes.keys()}

for idx in range(len(dataset_test)):
    image_path, label = dataset_test.samples[idx]
    if label in classes and len(class_indices[label]) < samples_per_class:
        img_name = os.path.basename(image_path)
        xml_name = os.path.splitext(img_name)[0] + ".xml"
        xml_path = os.path.join(bbox_dir, xml_name)

        if os.path.exists(xml_path):
            class_indices[label].append(idx)
    if all(len(indices) >= samples_per_class for indices in class_indices.values()):
        break

for cls, indices in class_indices.items():
    print(f"\nClass {cls} ({classes[cls]}): {len(indices)} candidate images")

    n_cols = 5
    n_rows = (len(indices) + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 3, n_rows * 3))
    axes = axes.flatten() if n_rows > 1 else [axes] if n_cols == 1 else axes

    for i, idx in enumerate(indices):
        image_tensor, label = dataset_test[idx]
        image_tensor = image_tensor.to(device)

        current_image = denormalize(image_tensor)
        current_image = current_image.permute(1, 2, 0).cpu().numpy()

        axes[i].imshow(current_image)
        axes[i].set_title(f"Index: {i}", fontsize=10)
        axes[i].axis('off')

    for i in range(len(indices), len(axes)):
        axes[i].axis('off')

    plt.tight_layout()
    plt.suptitle(f"Class {cls}: {classes[cls]} - Select indices to keep", fontsize=14, y=1.00)
    plt.show()

selected_indices = {0: [18, 2, 20, 8, 41, 43], 678: [8, 40, 43]}

filtered_class_indices = {cls: [class_indices[cls][i] for i in selected_indices[cls]] for cls in classes.keys()}

subsets = {}
loaders = {}

for cls, indices in class_indices.items():
    print(f"Class {cls} ({classes[cls]}): {len(indices)} selected images")
    subsets[cls] = Subset(dataset_test, indices)
    loaders[cls] = DataLoader(
        subsets[cls],
        batch_size=1,
        shuffle=False,
    )
for cls, indices in filtered_class_indices.items():
    print(f"Class {cls} ({classes[cls]}): {len(indices)} selected images")
    subsets[cls] = Subset(dataset_test, indices)
    loaders[cls] = DataLoader(
        subsets[cls],
        batch_size=1,
        shuffle=False,
    )

print("\nFiltered images with object regions masked:")
for cls in filtered_class_indices.keys():
    n_images = len(filtered_class_indices[cls])
    fig, axes = plt.subplots(2, n_images, figsize=(n_images * 3, 6))
    if n_images == 1:
        axes = axes.reshape(-1, 1)

    wnid = dataset_test.classes[cls]

    for i, idx in enumerate(filtered_class_indices[cls]):
        image_path, label = dataset_test.samples[idx]
        img_name = os.path.basename(image_path)
        xml_name = os.path.splitext(img_name)[0] + ".xml"
        xml_path = os.path.join(bbox_dir, xml_name)

        image_tensor, _ = dataset_test[idx]
        image_tensor = image_tensor.to(device)
        current_image = denormalize(image_tensor)
        current_image = current_image.permute(1, 2, 0).cpu().numpy()
        axes[0, i].imshow(current_image)
        axes[0, i].set_title("Original")
        axes[0, i].axis('off')

        masked_img = mask_object_regions(image_path, xml_path)
        masked_tensor = val_transform(masked_img).to(device)
        masked_display = denormalize(masked_tensor)
        masked_display = masked_display.permute(1, 2, 0).cpu().numpy()
        axes[1, i].imshow(masked_display)
        axes[1, i].set_title("Masked")
        axes[1, i].axis('off')

    plt.tight_layout()
    plt.suptitle(f"Class {cls}: {classes[cls]} - Original vs Masked", fontsize=14)
    plt.show()

In [ ]:
image_map = {step: {} for step in steps}
image_map["orig"] = {}
image_map["masked"] = {}
predictions = {step: {} for step in steps}

for step, model, linear in zip(steps, models_steps, linear_steps):
    model.to(device).eval()
    linear.to(device).eval()

    target_layers = [model.layer4[-1]]
    model_grad = nn.Sequential(model, linear)

    for cls, loader in loaders.items():
        print(f"Processing class {cls} ({classes[cls]}) for step {step}")

        predictions[step][cls] = {"orig": [], "masked": [], "orig_top5": [], "masked_top5": []}

        for index, (image_tensor, label) in enumerate(loader):
            label_item = label.item()
            image_tensor = image_tensor.to(device)

            current_image = denormalize(image_tensor.squeeze())
            current_image = current_image.permute(1, 2, 0).cpu().numpy()

            with torch.no_grad():
                logits = model_grad(image_tensor)
                pred_orig = logits.argmax(dim=1).item()
                top5_orig = logits.topk(5, dim=1)[1][0].cpu().tolist()
            predictions[step][cls]["orig"].append(pred_orig)
            predictions[step][cls]["orig_top5"].append(top5_orig)

            with HiResCAM(model=model_grad, target_layers=target_layers) as cam:
                grayscale_cam = cam(input_tensor=image_tensor, targets=None)
                grayscale_cam = grayscale_cam[0, :]
                visualization_orig = show_cam_on_image(current_image, grayscale_cam, use_rgb=True)

            idx = filtered_class_indices[cls][index]
            image_path, _ = dataset_test.samples[idx]
            img_name = os.path.basename(image_path)
            xml_name = os.path.splitext(img_name)[0] + ".xml"
            xml_path = os.path.join(bbox_dir, xml_name)

            masked_img = mask_object_regions(image_path, xml_path)
            masked_tensor = val_transform(masked_img).unsqueeze(0).to(device)

            masked_display = denormalize(masked_tensor.squeeze())
            masked_display = masked_display.permute(1, 2, 0).cpu().numpy()

            with torch.no_grad():
                logits_masked = model_grad(masked_tensor)
                pred_masked = logits_masked.argmax(dim=1).item()
                top5_masked = logits_masked.topk(5, dim=1)[1][0].cpu().tolist()
            predictions[step][cls]["masked"].append(pred_masked)
            predictions[step][cls]["masked_top5"].append(top5_masked)

            with HiResCAM(model=model_grad, target_layers=target_layers) as cam:
                grayscale_cam_masked = cam(input_tensor=masked_tensor, targets=None)
                grayscale_cam_masked = grayscale_cam_masked[0, :]
                visualization_masked = show_cam_on_image(masked_display, grayscale_cam_masked, use_rgb=True)

            if step == steps[0]:
                if label_item not in image_map["orig"]:
                    image_map["orig"][label_item] = []
                    image_map["masked"][label_item] = []
                image_map["orig"][label_item].append(to_pil_image(current_image))
                image_map["masked"][label_item].append(to_pil_image(masked_display))

            if label_item not in image_map[step]:
                image_map[step][label_item] = {"orig": [], "masked": []}
            image_map[step][label_item]["orig"].append(to_pil_image(visualization_orig))
            image_map[step][label_item]["masked"].append(to_pil_image(visualization_masked))

print("Processing complete!")

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

print(classes)
for cls in classes.keys():
    n_images = len(image_map["orig"][cls])
    n_rows = (len(steps) + 1) * 2

    fig, axes = plt.subplots(n_rows, n_images, figsize=(n_images * 3, n_rows * 2.5))

    if n_images == 1:
        axes = axes.reshape(-1, 1)

    for col in range(n_images):
        axes[0, col].imshow(image_map["orig"][cls][col])
        axes[0, col].axis('off')
        axes[0, col].set_title("Original", fontsize=10)
        if col == 0:
            axes[0, col].set_ylabel("Input\nImages", fontsize=12, rotation=0, ha='right', va='center')

        axes[1, col].imshow(image_map["masked"][cls][col])
        axes[1, col].axis('off')
        axes[1, col].set_title("Masked", fontsize=10)

        for step_idx, step in enumerate(steps):
            row_base = 2 + step_idx * 2

            pred_orig = predictions[step][cls]["orig"][col]
            top5_orig = predictions[step][cls]["orig_top5"][col]
            is_correct_orig = (pred_orig == cls)
            is_top5_orig = (cls in top5_orig)

            axes[row_base, col].imshow(image_map[step][cls]["orig"][col])
            axes[row_base, col].axis('off')

            border_color = 'green' if is_correct_orig else ('orange' if is_top5_orig else 'red')
            rect = Rectangle((0, 0), 1, 1, transform=axes[row_base, col].transAxes,
                           linewidth=3, edgecolor=border_color, facecolor='none')
            axes[row_base, col].add_patch(rect)

            title_text = f"{'✓' if is_correct_orig else ('⊙' if is_top5_orig else '✗')} (pred: {pred_orig})"
            axes[row_base, col].set_title(title_text, fontsize=9, color=border_color, weight='bold')

            if col == 0:
                axes[row_base, col].set_ylabel(f"Step {step}\n(Orig)", fontsize=11, rotation=0, ha='right', va='center')

            pred_masked = predictions[step][cls]["masked"][col]
            top5_masked = predictions[step][cls]["masked_top5"][col]
            is_correct_masked = (pred_masked == cls)
            is_top5_masked = (cls in top5_masked)

            axes[row_base + 1, col].imshow(image_map[step][cls]["masked"][col])
            axes[row_base + 1, col].axis('off')

            border_color = 'green' if is_top5_masked else 'red'
            rect = Rectangle((0, 0), 1, 1, transform=axes[row_base + 1, col].transAxes,
                           linewidth=3, edgecolor=border_color, facecolor='none')
            axes[row_base + 1, col].add_patch(rect)

            title_text = f"{'✓' if is_top5_masked else '✗'} (pred: {pred_masked})"
            axes[row_base + 1, col].set_title(title_text, fontsize=9, color=border_color, weight='bold')

            if col == 0:
                axes[row_base + 1, col].set_ylabel(f"Step {step}\n(Masked)", fontsize=11, rotation=0, ha='right', va='center')

    plt.tight_layout()
    plt.suptitle(f"Class {cls}: {classes[cls]} - Original vs Masked GradCAM Comparison",
                 fontsize=16, y=0.998)
    plt.show()

    print(f"\n=== Class {cls} ({classes[cls]}) Accuracy Summary ===")
    for step in steps:
        orig_top1 = sum(1 for p in predictions[step][cls]["orig"] if p == cls)
        masked_top1 = sum(1 for p in predictions[step][cls]["masked"] if p == cls)
        orig_top5 = sum(1 for top5 in predictions[step][cls]["orig_top5"] if cls in top5)
        masked_top5 = sum(1 for top5 in predictions[step][cls]["masked_top5"] if cls in top5)
        total = len(predictions[step][cls]["orig"])

In [ ]:
from matplotlib.patches import Rectangle

# 1. Mask the background while retaining the object.
def mask_background_regions(image_path, xml_path, fill_color=(128, 128, 128)):
    """Mask the background while retaining the object region."""
    from PIL import Image
    import xml.etree.ElementTree as ET

    img = Image.open(image_path).convert('RGB')
    img_array = np.array(img)

    # Build the mask: background=True, object=False.
    mask = np.ones(img_array.shape[:2], dtype=bool)

    tree = ET.parse(xml_path)
    root = tree.getroot()

    for obj in root.findall('.//object'):
        bndbox = obj.find('bndbox')
        xmin = int(bndbox.find('xmin').text)
        ymin = int(bndbox.find('ymin').text)
        xmax = int(bndbox.find('xmax').text)
        ymax = int(bndbox.find('ymax').text)

        # Leave the object region unmasked.
        mask[ymin:ymax, xmin:xmax] = False

    # Fill the background with fill_color.
    result = img_array.copy()
    result[mask] = fill_color

    return Image.fromarray(result)


# 2. Evaluate all three conditions and collect images.
image_map = {step: {} for step in steps}
image_map["orig"] = {}
image_map["masked_obj"] = {}  # Mask the object.
image_map["masked_bg"] = {}   # Mask the background.

predictions = {step: {} for step in steps}

for step, model, linear in zip(steps, models_steps, linear_steps):
    model.to(device).eval()
    linear.to(device).eval()

    target_layers = [model.layer4[-1]]
    model_grad = nn.Sequential(model, linear)

    for cls, loader in loaders.items():
        print(f"Processing class {cls} ({classes[cls]}) for step {step}")

        predictions[step][cls] = {
            "orig": [], "orig_top5": [],
            "masked_obj": [], "masked_obj_top5": [],
            "masked_bg": [], "masked_bg_top5": []
        }

        for index, (image_tensor, label) in enumerate(loader):
            label_item = label.item()
            image_tensor = image_tensor.to(device)

            # Process the original image.
            current_image = denormalize(image_tensor.squeeze())
            current_image = current_image.permute(1, 2, 0).cpu().numpy()

            with torch.no_grad():
                logits = model_grad(image_tensor)
                pred_orig = logits.argmax(dim=1).item()
                top5_orig = logits.topk(5, dim=1)[1][0].cpu().tolist()
            predictions[step][cls]["orig"].append(pred_orig)
            predictions[step][cls]["orig_top5"].append(top5_orig)

            # GradCAM for original
            with HiResCAM(model=model_grad, target_layers=target_layers) as cam:
                grayscale_cam = cam(input_tensor=image_tensor, targets=None)
                grayscale_cam = grayscale_cam[0, :]
                visualization_orig = show_cam_on_image(current_image, grayscale_cam, use_rgb=True)

            # Get the file path.
            idx = class_indices[cls][index]
            image_path, _ = dataset_test.samples[idx]
            img_name = os.path.basename(image_path)
            xml_name = os.path.splitext(img_name)[0] + ".xml"
            xml_path = os.path.join(bbox_dir, xml_name)

            # Process the image with the object masked.
            masked_obj_img = mask_object_regions(image_path, xml_path)
            masked_obj_tensor = val_transform(masked_obj_img).unsqueeze(0).to(device)
            masked_obj_display = denormalize(masked_obj_tensor.squeeze())
            masked_obj_display = masked_obj_display.permute(1, 2, 0).cpu().numpy()

            with torch.no_grad():
                logits_masked_obj = model_grad(masked_obj_tensor)
                pred_masked_obj = logits_masked_obj.argmax(dim=1).item()
                top5_masked_obj = logits_masked_obj.topk(5, dim=1)[1][0].cpu().tolist()
            predictions[step][cls]["masked_obj"].append(pred_masked_obj)
            predictions[step][cls]["masked_obj_top5"].append(top5_masked_obj)

            with HiResCAM(model=model_grad, target_layers=target_layers) as cam:
                grayscale_cam_obj = cam(input_tensor=masked_obj_tensor, targets=None)
                grayscale_cam_obj = grayscale_cam_obj[0, :]
                visualization_masked_obj = show_cam_on_image(masked_obj_display, grayscale_cam_obj, use_rgb=True)

            # Process the image with the background masked, retaining only the object.
            masked_bg_img = mask_background_regions(image_path, xml_path)
            masked_bg_tensor = val_transform(masked_bg_img).unsqueeze(0).to(device)
            masked_bg_display = denormalize(masked_bg_tensor.squeeze())
            masked_bg_display = masked_bg_display.permute(1, 2, 0).cpu().numpy()

            with torch.no_grad():
                logits_masked_bg = model_grad(masked_bg_tensor)
                pred_masked_bg = logits_masked_bg.argmax(dim=1).item()
                top5_masked_bg = logits_masked_bg.topk(5, dim=1)[1][0].cpu().tolist()
            predictions[step][cls]["masked_bg"].append(pred_masked_bg)
            predictions[step][cls]["masked_bg_top5"].append(top5_masked_bg)

            with HiResCAM(model=model_grad, target_layers=target_layers) as cam:
                grayscale_cam_bg = cam(input_tensor=masked_bg_tensor, targets=None)
                grayscale_cam_bg = grayscale_cam_bg[0, :]
                visualization_masked_bg = show_cam_on_image(masked_bg_display, grayscale_cam_bg, use_rgb=True)

            # Save original and masked images at the first step only.
            if step == steps[0]:
                if label_item not in image_map["orig"]:
                    image_map["orig"][label_item] = []
                    image_map["masked_obj"][label_item] = []
                    image_map["masked_bg"][label_item] = []
                image_map["orig"][label_item].append(to_pil_image(current_image))
                image_map["masked_obj"][label_item].append(to_pil_image(masked_obj_display))
                image_map["masked_bg"][label_item].append(to_pil_image(masked_bg_display))

            # Save the GradCAM visualization.
            if label_item not in image_map[step]:
                image_map[step][label_item] = {"orig": [], "masked_obj": [], "masked_bg": []}
            image_map[step][label_item]["orig"].append(to_pil_image(visualization_orig))
            image_map[step][label_item]["masked_obj"].append(to_pil_image(visualization_masked_obj))
            image_map[step][label_item]["masked_bg"].append(to_pil_image(visualization_masked_bg))

print("Processing complete!")


# 3. Filter and visualize images satisfying the conditions.
# Condition: original correct, masked object incorrect, masked background correct.

for cls in classes.keys():
    print(f"\n=== Class {cls} ({classes[cls]}) ===")

    # Find image indices satisfying the conditions at each step.
    # Filter at the final step, or select a specific step.
    reference_step = steps[-1]  # Or choose another step.

    valid_indices = []
    for i in range(len(predictions[reference_step][cls]["orig"])):
        orig_correct = (predictions[reference_step][cls]["orig"][i] == cls)
        masked_obj_wrong = (predictions[reference_step][cls]["masked_obj"][i] != cls)
        masked_bg_correct = (predictions[reference_step][cls]["masked_bg"][i] == cls)

        if orig_correct and masked_obj_wrong and masked_bg_correct:
            valid_indices.append(i)

    print(f"Found {len(valid_indices)} images satisfying all conditions at step {reference_step}")
    print(f"Indices: {valid_indices}")

    if len(valid_indices) == 0:
        print("No images found. Trying with top-5 criteria...")
        # Use the Top-5 criterion.
        for i in range(len(predictions[reference_step][cls]["orig"])):
            orig_correct = (predictions[reference_step][cls]["orig"][i] == cls)
            masked_obj_wrong = (cls not in predictions[reference_step][cls]["masked_obj_top5"][i])
            masked_bg_correct = (cls in predictions[reference_step][cls]["masked_bg_top5"][i])

            if orig_correct and masked_obj_wrong and masked_bg_correct:
                valid_indices.append(i)
        print(f"Found {len(valid_indices)} images with relaxed criteria")

    if len(valid_indices) == 0:
        continue

    # Visualization
    n_images = len(valid_indices)
    n_rows = 3 + len(steps) * 3  # Three input rows, followed by three GradCAM rows per step.

    fig, axes = plt.subplots(n_rows, n_images, figsize=(n_images * 3, n_rows * 2.5))

    if n_images == 1:
        axes = axes.reshape(-1, 1)

    for col, img_idx in enumerate(valid_indices):
        # Row 0-2: Input images (Original, Masked Object, Masked Background)
        axes[0, col].imshow(image_map["orig"][cls][img_idx])
        axes[0, col].axis('off')
        axes[0, col].set_title("Original", fontsize=10)
        if col == 0:
            axes[0, col].set_ylabel("Input\nOriginal", fontsize=11, rotation=0, ha='right', va='center')

        axes[1, col].imshow(image_map["masked_obj"][cls][img_idx])
        axes[1, col].axis('off')
        axes[1, col].set_title("Masked Object", fontsize=10)
        if col == 0:
            axes[1, col].set_ylabel("Input\nMasked Obj", fontsize=11, rotation=0, ha='right', va='center')

        axes[2, col].imshow(image_map["masked_bg"][cls][img_idx])
        axes[2, col].axis('off')
        axes[2, col].set_title("Masked BG", fontsize=10)
        if col == 0:
            axes[2, col].set_ylabel("Input\nMasked BG", fontsize=11, rotation=0, ha='right', va='center')

        # GradCAM at each step
        for step_idx, step in enumerate(steps):
            row_base = 3 + step_idx * 3

            # Original GradCAM
            pred = predictions[step][cls]["orig"][img_idx]
            is_correct = (pred == cls)
            border_color = 'green' if is_correct else 'red'

            axes[row_base, col].imshow(image_map[step][cls]["orig"][img_idx])
            axes[row_base, col].axis('off')
            rect = Rectangle((0, 0), 1, 1, transform=axes[row_base, col].transAxes,
                           linewidth=3, edgecolor=border_color, facecolor='none')
            axes[row_base, col].add_patch(rect)
            axes[row_base, col].set_title(f"{'✓' if is_correct else '✗'} pred:{pred}", fontsize=9, color=border_color)
            if col == 0:
                axes[row_base, col].set_ylabel(f"Step {step}\nOrig", fontsize=10, rotation=0, ha='right', va='center')

            # Masked Object GradCAM
            pred = predictions[step][cls]["masked_obj"][img_idx]
            is_correct = (pred == cls)
            border_color = 'green' if is_correct else 'red'

            axes[row_base + 1, col].imshow(image_map[step][cls]["masked_obj"][img_idx])
            axes[row_base + 1, col].axis('off')
            rect = Rectangle((0, 0), 1, 1, transform=axes[row_base + 1, col].transAxes,
                           linewidth=3, edgecolor=border_color, facecolor='none')
            axes[row_base + 1, col].add_patch(rect)
            axes[row_base + 1, col].set_title(f"{'✓' if is_correct else '✗'} pred:{pred}", fontsize=9, color=border_color)
            if col == 0:
                axes[row_base + 1, col].set_ylabel(f"Step {step}\nMskObj", fontsize=10, rotation=0, ha='right', va='center')

            # Masked Background GradCAM
            pred = predictions[step][cls]["masked_bg"][img_idx]
            is_correct = (pred == cls)
            border_color = 'green' if is_correct else 'red'

            axes[row_base + 2, col].imshow(image_map[step][cls]["masked_bg"][img_idx])
            axes[row_base + 2, col].axis('off')
            rect = Rectangle((0, 0), 1, 1, transform=axes[row_base + 2, col].transAxes,
                           linewidth=3, edgecolor=border_color, facecolor='none')
            axes[row_base + 2, col].add_patch(rect)
            axes[row_base + 2, col].set_title(f"{'✓' if is_correct else '✗'} pred:{pred}", fontsize=9, color=border_color)
            if col == 0:
                axes[row_base + 2, col].set_ylabel(f"Step {step}\nMskBG", fontsize=10, rotation=0, ha='right', va='center')

    plt.tight_layout()
    plt.suptitle(f"Class {cls}: {classes[cls]}\nCondition: Orig ✓, MaskedObj ✗, MaskedBG ✓",
                 fontsize=14, y=1.01)
    plt.show()